# How to create git workspace in Snowflake?
In Snowflake, A git workspace is dedicated to only one git repository. It allows us to create new branch, fetch the changes, visual interface to see modified files, new added files and deleted files.
We can commit and push changes to remote.

### How to clone git repository?
- Go to snowsight UI
- `Projects` tab in left sidebar
- Click on `Workspaces` option
- Click on `Create workspace`
- Click on `Git workspace`
- You need to create two things (Use same UI or write SQL command for this):
    - API Integration object
    - Credentials secret object

When we clone git repository using workspace, it is not getting registered in Schema.

When we clone git repository using SQL command, it is registered in schema. But it has below limitations:
- We can not interact with it in workspace.
- It is visible in Database explorer, still remains useless.
- To see the files, you need to go to catalog explorer, then locate your repository in schema.
- It is always read-only. You cannot commit any changes.
- We can view and execute code of sql files only.
- To view and execute sql code, click on sql file and tap `execute immediate`

# Git in Snowflake limitations
This topic describes limitations for using Git repositories from within Snowflake.

- Currently, only the following Snowflake features can write to the repository:
    - Workspaces
    - Streamlit applications
    - Notebooks
      
    For other Snowflake code, access to the repository is read-only.

- When you connect to a Git repository using a workspace, the following limitation applies:
    - The Git repository can’t be empty. It must have at least one commit.
    - Creating a local Git repository in Snowflake is supported only when using the Workspaces user interface to create it.
    - It isn’t supported when you create the repository by using `CREATE GIT REPOSITORY` in a workspace. This is because when using the SQL command, the flow does not include presenting a user interface with which to sign in.

- Sharing Snowflake Git repository clones is not supported through data sharing or apps built on the Snowflake Native App Framework.

- Creating Snowflake Git repository clones inside application packages is not supported and might be blocked in the future.

- Creating Snowflake Git repository clones inside native applications on the consumer side is not supported.

- Snowflake doesn’t currently support submodules, so you won’t be able to see submodule files. Snowflake won’t download those files from the remote repository nor upload them to the remote repository.

- Git repositories larger than 2 GB aren’t supported.

In [ ]:
%%sql -r dataframe_3
CREATE DATABASE REPO_DATABASE
COMMENT = 'This database is created to manages repositories';

CREATE SCHEMA REPO_DATABASE.GIT_REPO_SCHEMA
COMMENT = 'This database is created to manages Github Clones';

In [ ]:
%%sql -r dataframe_2
CREATE OR REPLACE SECRET REPO_DATABASE.GIT_REPO_SCHEMA.my_git_secret
  TYPE = password
  USERNAME = 'Ajay-Kumar-Maurya' -- github username
  PASSWORD = '<your-github-pat>' --  -- github token
  COMMENT = 'This secret stores github Personal access tokens (classic)';

CREATE OR REPLACE API INTEGRATION github_api_integration
  API_PROVIDER = git_https_api
  API_ALLOWED_PREFIXES = ('https://github.com/Ajay-Kumar-Maurya')
  ALLOWED_AUTHENTICATION_SECRETS = ( REPO_DATABASE.GIT_REPO_SCHEMA.my_git_secret )
  ENABLED = TRUE
  COMMENT = 'Github API Integration using PAT (Classic) to clone git-repository';

In [ ]:
%%sql -r dataframe_4
-- We can't create tag directly in sql command. First, create them and then use them.
CREATE TAG git_provider COMMENT = 'Git provider name';
CREATE TAG purpose COMMENT = 'Repository purpose';

In [ ]:
%%sql -r dataframe_1
-- Creates a Snowflake Git repository clone in the schema.
CREATE OR REPLACE GIT REPOSITORY REPO_DATABASE.GIT_REPO_SCHEMA.sf_git_clone
  ORIGIN = 'https://github.com/Ajay-Kumar-Maurya/snowflake-notes.git'
  API_INTEGRATION = github_api_integration
  COMMENT = 'This is cloned repository that contains notes about snowflake'
  WITH TAG ( git_provider = 'GitHub' , purpose = 'Snowflake-notes' );

In [ ]:
%%sql -r dataframe_6
SHOW GIT BRANCHES IN GIT REPOSITORY REPO_DATABASE.GIT_REPO_SCHEMA.sf_git_clone;

In [ ]:
%%sql -r dataframe_7
SHOW GIT REPOSITORIES IN SCHEMA REPO_DATABASE.GIT_REPO_SCHEMA;

In [ ]:
ALTER GIT REPOSITORY REPO_DATABASE.GIT_REPO_SCHEMA.sf_git_clone FETCH;

In [ ]:
-- List by branch name
LS @REPO_DATABASE.GIT_REPO_SCHEMA.sf_git_clone/branches/<branch_name>;

-- List by commit hash
LS @REPO_DATABASE.GIT_REPO_SCHEMA.sf_git_clone/commits/<commit_hash>;

In [ ]:
EXECUTE IMMEDIATE FROM @snowflake_extensions/branches/main/sql/create-database.sql;

/* Syntax
EXECUTE IMMEDIATE FROM @snowflake_extensions/branches/<branch_name>/<path1>/<path2>/<file-name>.sql;

This command only works with sql files.
*/